# Credit Default Swap Pricing and Risk

This notebook builds a transparent first model for a **single-name credit default swap (CDS)**. It is divided into short parts so each concept can be discussed independently.

The notebook includes:

- deterministic discount factors;
- piecewise-constant hazard rates and survival probabilities;
- premium-leg and protection-leg valuation;
- accrued premium on default;
- hazard-curve bootstrapping from par CDS spreads;
- off-market CDS present value;
- parallel and key-rate CS01;
- discount-curve IR01, recovery sensitivity, and jump-to-default;
- full-revaluation credit scenarios;
- numerical and theoretical validation.

> **Data status:** all spreads and rates below are illustrative. Live market data is not required to understand or test the model. Production use requires contractual conventions, calibrated discount and credit curves, recovery governance, and independent validation.


## Part 1 — Contract and model scope

A CDS protection buyer pays periodic premium and receives a loss payment if the reference entity defaults. A protection seller receives premium and owes the loss payment after default.

This first implementation uses a reduced-form, risk-neutral model with the following controlled assumptions:

1. valuation occurs on a coupon date, so there is no accrued premium at time zero;
2. quoted CDS spreads are running par spreads with no upfront payment;
3. coupon dates are equally spaced year fractions, without calendars or stubs;
4. hazard rates are piecewise constant between quoted maturities;
5. recovery is constant and exogenous;
6. interest rates, default intensity, and recovery are deterministic and independent;
7. protection and accrued premium on default are discounted at each interval midpoint;
8. counterparty risk, wrong-way risk, restructuring details, settlement delays, and cheapest-to-deliver options are outside this version.

The model value is reported from the selected side's perspective. A positive number is an asset to that side.


In [ ]:
from dataclasses import dataclass, replace
import math
from typing import Literal

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.options.display.float_format = lambda x: f'{x:,.6f}'

## Part 2 — Contract, market quotes, and discount curve

Rates, running spreads, and hazard rates are decimal annualized inputs: 100 bp is entered as `0.01`. Recovery is a fraction of par.

The discount curve uses continuously compounded zero rates:

$$D(0,t)=e^{-z(t)t}.$$

Zero rates are linearly interpolated between supplied pillars with flat-zero extrapolation outside the final pillar.


In [ ]:
@dataclass(frozen=True)
class CDSContract:
    maturity: float
    coupon: float
    notional: float
    side: Literal['protection_buyer', 'protection_seller']
    payment_frequency: int = 4

    def validate(self):
        numeric = [self.maturity, self.coupon, self.notional]
        if not all(math.isfinite(float(x)) for x in numeric):
            raise ValueError('All contract inputs must be finite.')
        if self.maturity <= 0 or self.notional <= 0 or self.coupon < 0:
            raise ValueError('Maturity and notional must be positive; coupon cannot be negative.')
        if self.side.lower() not in {'protection_buyer', 'protection_seller'}:
            raise ValueError("side must be 'protection_buyer' or 'protection_seller'.")
        if not isinstance(self.payment_frequency, int) or self.payment_frequency <= 0:
            raise ValueError('payment_frequency must be a positive integer.')
        periods = self.maturity * self.payment_frequency
        if abs(periods - round(periods)) > 1e-10:
            raise ValueError('maturity × payment_frequency must be an integer in this simplified schedule.')


@dataclass(frozen=True)
class CDSMarketQuotes:
    maturities: tuple
    par_spreads: tuple
    recovery: float
    payment_frequency: int = 4

    def __post_init__(self):
        maturities = tuple(float(x) for x in self.maturities)
        spreads = tuple(float(x) for x in self.par_spreads)
        object.__setattr__(self, 'maturities', maturities)
        object.__setattr__(self, 'par_spreads', spreads)
        if len(maturities) != len(spreads) or len(maturities) == 0:
            raise ValueError('Quote maturities and spreads must have equal non-zero length.')
        if any(t <= 0 for t in maturities) or any(b <= a for a, b in zip(maturities, maturities[1:])):
            raise ValueError('Quote maturities must be positive and strictly increasing.')
        if any(s < 0 or not math.isfinite(s) for s in spreads):
            raise ValueError('Quoted spreads must be finite and non-negative.')
        if not 0.0 <= self.recovery < 1.0:
            raise ValueError('Recovery must lie in [0, 1).')
        if not isinstance(self.payment_frequency, int) or self.payment_frequency <= 0:
            raise ValueError('payment_frequency must be a positive integer.')
        for maturity in maturities:
            periods = maturity * self.payment_frequency
            if abs(periods - round(periods)) > 1e-10:
                raise ValueError('Every quote maturity must align with the simplified coupon schedule.')


@dataclass(frozen=True)
class ZeroCurve:
    name: str
    times: tuple
    rates: tuple

    def __post_init__(self):
        times = tuple(float(x) for x in self.times)
        rates = tuple(float(x) for x in self.rates)
        object.__setattr__(self, 'times', times)
        object.__setattr__(self, 'rates', rates)
        if len(times) != len(rates) or len(times) < 2:
            raise ValueError('Curve times and rates must have equal length of at least two.')
        if abs(times[0]) > 1e-14 or any(t < 0 for t in times):
            raise ValueError('The first curve time must be 0 and all times must be non-negative.')
        if any(b <= a for a, b in zip(times, times[1:])):
            raise ValueError('Curve times must be strictly increasing.')
        if not all(math.isfinite(x) for x in times + rates):
            raise ValueError('Curve inputs must be finite.')

    def zero_rate(self, time):
        x = np.asarray(time, dtype=float)
        if np.any(x < 0):
            raise ValueError('Discount times cannot be negative.')
        value = np.interp(x, self.times, self.rates, left=self.rates[0], right=self.rates[-1])
        return float(value) if x.ndim == 0 else value

    def discount(self, time):
        x = np.asarray(time, dtype=float)
        value = np.exp(-np.asarray(self.zero_rate(x)) * x)
        return float(value) if x.ndim == 0 else value

    def parallel_shift(self, basis_points):
        shift = float(basis_points) / 10_000.0
        return replace(self, rates=tuple(rate + shift for rate in self.rates))


@dataclass(frozen=True)
class CDSResult:
    present_value: float
    protection_leg: float
    scheduled_premium_leg: float
    accrual_on_default: float
    risky_pv01: float
    fair_spread: float

## Part 3 — Hazard rates, survival, and default probability

The hazard rate $\lambda(t)$ is a risk-neutral instantaneous default intensity. It is not itself a default probability.

The cumulative hazard is

$$H(t)=\int_0^t\lambda(u)\,du,$$

and survival probability is

$$Q(t)=\Pr(\tau>t)=e^{-H(t)}.$$

The probability of default during $(t_{i-1},t_i]$ is

$$\Delta PD_i=Q(t_{i-1})-Q(t_i).$$

This notebook models $\lambda(t)$ as constant inside each quoted maturity segment. The last segment's hazard rate is extrapolated flat beyond its endpoint.


In [ ]:
@dataclass(frozen=True)
class HazardCurve:
    segment_ends: tuple
    hazard_rates: tuple

    def __post_init__(self):
        ends = tuple(float(x) for x in self.segment_ends)
        hazards = tuple(float(x) for x in self.hazard_rates)
        object.__setattr__(self, 'segment_ends', ends)
        object.__setattr__(self, 'hazard_rates', hazards)
        if len(ends) != len(hazards) or len(ends) == 0:
            raise ValueError('Segment ends and hazard rates must have equal non-zero length.')
        if any(t <= 0 for t in ends) or any(b <= a for a, b in zip(ends, ends[1:])):
            raise ValueError('Hazard segment ends must be positive and strictly increasing.')
        if any(h < 0 or not math.isfinite(h) for h in hazards):
            raise ValueError('Hazard rates must be finite and non-negative.')

    def cumulative_hazard(self, time):
        x = np.asarray(time, dtype=float)
        if np.any(x < 0):
            raise ValueError('Survival times cannot be negative.')
        result = np.zeros_like(x, dtype=float)
        start = 0.0
        for end, hazard in zip(self.segment_ends, self.hazard_rates):
            result += hazard * np.clip(x - start, 0.0, end - start)
            start = end
        result += self.hazard_rates[-1] * np.maximum(x - self.segment_ends[-1], 0.0)
        return float(result) if x.ndim == 0 else result

    def survival(self, time):
        value = np.exp(-np.asarray(self.cumulative_hazard(time)))
        return float(value) if np.asarray(time).ndim == 0 else value

## Part 4 — Premium and protection legs

Let $t_i$ be coupon dates, $m_i$ interval midpoints, $\alpha_i$ coupon accrual fractions, and $R$ recovery. The protection leg per unit notional is approximated by

$$PL=(1-R)\sum_i D(0,m_i)[Q(t_{i-1})-Q(t_i)].$$

The scheduled premium annuity is

$$RA_{scheduled}=\sum_i\alpha_iD(0,t_i)Q(t_i).$$

Because default can occur between coupon dates, the protection buyer normally owes accrued premium up to default. Using the half-period approximation,

$$RA_{accrual}=\sum_i\frac{\alpha_i}{2}D(0,m_i)[Q(t_{i-1})-Q(t_i)].$$

For contractual coupon $c$, buyer-side value is

$$V_{buyer}=N[PL-c(RA_{scheduled}+RA_{accrual})].$$

The seller value is its negative.


In [ ]:
def cds_schedule(maturity: float, payment_frequency: int) -> pd.DataFrame:
    periods_float = maturity * payment_frequency
    if maturity <= 0 or payment_frequency <= 0 or abs(periods_float - round(periods_float)) > 1e-10:
        raise ValueError('Maturity must be positive and align with the payment frequency.')
    periods = int(round(periods_float))
    accrual = 1.0 / payment_frequency
    starts = np.arange(periods, dtype=float) * accrual
    ends = np.arange(1, periods + 1, dtype=float) * accrual
    return pd.DataFrame({
        'Period': np.arange(1, periods + 1),
        'Start time': starts,
        'End time': ends,
        'Midpoint': 0.5 * (starts + ends),
        'Accrual fraction': np.full(periods, accrual),
    })


def cds_leg_factors(
    maturity: float,
    payment_frequency: int,
    discount_curve: ZeroCurve,
    hazard_curve: HazardCurve,
):
    detail = cds_schedule(maturity, payment_frequency)
    starts = detail['Start time'].to_numpy()
    ends = detail['End time'].to_numpy()
    midpoints = detail['Midpoint'].to_numpy()
    accruals = detail['Accrual fraction'].to_numpy()

    survival_start = hazard_curve.survival(starts)
    survival_end = hazard_curve.survival(ends)
    default_probability = survival_start - survival_end
    discount_payment = discount_curve.discount(ends)
    discount_default = discount_curve.discount(midpoints)

    scheduled_annuity = float(np.sum(accruals * discount_payment * survival_end))
    accrual_on_default_annuity = float(
        np.sum(0.5 * accruals * discount_default * default_probability)
    )
    default_annuity = float(np.sum(discount_default * default_probability))

    detail['Survival at start'] = survival_start
    detail['Survival at end'] = survival_end
    detail['Interval default probability'] = default_probability
    detail['Payment discount factor'] = discount_payment
    detail['Default discount factor'] = discount_default
    return scheduled_annuity, accrual_on_default_annuity, default_annuity, detail


def cds_par_spread(
    maturity: float,
    payment_frequency: int,
    discount_curve: ZeroCurve,
    hazard_curve: HazardCurve,
    recovery: float,
) -> float:
    if not 0.0 <= recovery < 1.0:
        raise ValueError('Recovery must lie in [0, 1).')
    scheduled, accrual_default, default_annuity, _ = cds_leg_factors(
        maturity, payment_frequency, discount_curve, hazard_curve
    )
    risky_annuity = scheduled + accrual_default
    if risky_annuity <= 0:
        raise ValueError('Risky premium annuity must be positive.')
    return (1.0 - recovery) * default_annuity / risky_annuity


def price_cds(
    contract: CDSContract,
    discount_curve: ZeroCurve,
    hazard_curve: HazardCurve,
    recovery: float,
) -> CDSResult:
    contract.validate()
    if not 0.0 <= recovery < 1.0:
        raise ValueError('Recovery must lie in [0, 1).')
    scheduled, accrual_default, default_annuity, _ = cds_leg_factors(
        contract.maturity, contract.payment_frequency, discount_curve, hazard_curve
    )
    risky_annuity = scheduled + accrual_default
    protection_leg = contract.notional * (1.0 - recovery) * default_annuity
    scheduled_premium_leg = contract.notional * contract.coupon * scheduled
    accrual_on_default = contract.notional * contract.coupon * accrual_default
    buyer_value = protection_leg - scheduled_premium_leg - accrual_on_default
    side_sign = 1.0 if contract.side.lower() == 'protection_buyer' else -1.0
    fair_spread = (1.0 - recovery) * default_annuity / risky_annuity
    return CDSResult(
        present_value=side_sign * buyer_value,
        protection_leg=protection_leg,
        scheduled_premium_leg=scheduled_premium_leg,
        accrual_on_default=accrual_on_default,
        risky_pv01=contract.notional * risky_annuity * 0.0001,
        fair_spread=fair_spread,
    )

## Part 5 — Bootstrapping the hazard curve

A quoted par spread is the coupon that makes a new CDS worth zero:

$$s_{par}=\frac{(1-R)\sum_iD(0,m_i)\Delta PD_i}{RA_{scheduled}+RA_{accrual}}.$$

The model solves hazard rates sequentially. The first quote determines the first hazard segment. Each later quote determines one new segment while earlier calibrated hazards remain fixed.

Hazard rates are model-implied risk-neutral parameters. They depend on discounting, recovery, timing assumptions, and the complete spread term structure. The approximation $\lambda\approx s/(1-R)$ is useful intuition but is not the bootstrap used here.


In [ ]:
def bootstrap_hazard_curve(
    quotes: CDSMarketQuotes,
    discount_curve: ZeroCurve,
    spread_tolerance: float = 1e-13,
    max_iterations: int = 200,
) -> HazardCurve:
    hazards = []
    for index, (maturity, market_spread) in enumerate(
        zip(quotes.maturities, quotes.par_spreads)
    ):
        segment_ends = quotes.maturities[: index + 1]

        def error(candidate_hazard):
            curve = HazardCurve(segment_ends, tuple(hazards + [candidate_hazard]))
            model_spread = cds_par_spread(
                maturity, quotes.payment_frequency, discount_curve, curve, quotes.recovery
            )
            return model_spread - market_spread

        low = 0.0
        low_error = error(low)
        if low_error > spread_tolerance:
            raise ValueError(
                f'Quote at {maturity:g}Y requires a negative hazard in this model. '
                'Review spread consistency, recovery, and conventions.'
            )
        if abs(low_error) <= spread_tolerance:
            hazards.append(0.0)
            continue

        high = max(0.05, 3.0 * market_spread / max(1.0 - quotes.recovery, 1e-12))
        high_error = error(high)
        while high_error < 0.0 and high < 20.0:
            high *= 2.0
            high_error = error(high)
        if high_error < 0.0:
            raise ValueError(f'Could not bracket the hazard rate for the {maturity:g}Y quote.')

        for _ in range(max_iterations):
            middle = 0.5 * (low + high)
            middle_error = error(middle)
            if abs(middle_error) <= spread_tolerance:
                low = high = middle
                break
            if middle_error > 0.0:
                high = middle
            else:
                low = middle
        hazards.append(0.5 * (low + high))

    return HazardCurve(quotes.maturities, tuple(hazards))

## Part 6 — Illustrative market data and calibrated credit curve

The sample market contains quarterly-pay par spreads at 1Y, 3Y, 5Y, 7Y, and 10Y with a 40% recovery assumption. The spread curve is upward sloping.

The chart displays three different objects:

- quoted par spread;
- calibrated segment hazard rate;
- cumulative default probability $1-Q(t)$.

They should not be interpreted as interchangeable.


In [ ]:
discount_curve = ZeroCurve(
    name='Illustrative OIS discount curve',
    times=(0.0, 0.5, 1.0, 2.0, 3.0, 5.0, 7.0, 10.0),
    rates=(0.0350, 0.0360, 0.0370, 0.0390, 0.0400, 0.0415, 0.0420, 0.0425),
)
market_quotes = CDSMarketQuotes(
    maturities=(1.0, 3.0, 5.0, 7.0, 10.0),
    par_spreads=(0.0080, 0.0110, 0.0140, 0.0165, 0.0190),
    recovery=0.40,
    payment_frequency=4,
)
hazard_curve = bootstrap_hazard_curve(market_quotes, discount_curve)

credit_profile = pd.DataFrame({
    'Maturity (years)': market_quotes.maturities,
    'Market par spread': market_quotes.par_spreads,
    'Segment hazard rate': hazard_curve.hazard_rates,
    'Survival probability': hazard_curve.survival(market_quotes.maturities),
})
credit_profile['Cumulative default probability'] = 1.0 - credit_profile['Survival probability']
display(credit_profile)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(credit_profile['Maturity (years)'], 10_000 * credit_profile['Market par spread'], marker='o', label='Par spread')
axes[0].plot(credit_profile['Maturity (years)'], 10_000 * credit_profile['Segment hazard rate'], marker='o', label='Hazard rate')
axes[0].set_xlabel('Maturity (years)')
axes[0].set_ylabel('Basis points per year')
axes[0].set_title('Spread quotes and hazard segments')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(credit_profile['Maturity (years)'], 100 * credit_profile['Cumulative default probability'], marker='o')
axes[1].set_xlabel('Maturity (years)')
axes[1].set_ylabel('Risk-neutral cumulative default probability (%)')
axes[1].set_title('Implied cumulative default probability')
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Part 7 — Baseline CDS valuation

The sample trade buys five-year protection on 10 million notional and pays a 100 bp contractual coupon. Because the illustrative five-year market par spread is 140 bp, the below-market coupon should make the buyer-side present value positive.

Risky PV01 is the present value of one basis point of contractual premium. Under this model and the same calibrated curve,

$$V_{buyer}=(s_{par}-c)\times\frac{RiskyPV01}{1\text{ bp}}.$$


In [ ]:
contract = CDSContract(
    maturity=5.0,
    coupon=0.0100,
    notional=10_000_000.0,
    side='protection_buyer',
    payment_frequency=4,
)
result = price_cds(contract, discount_curve, hazard_curve, market_quotes.recovery)
_, _, _, cashflow_detail = cds_leg_factors(
    contract.maturity, contract.payment_frequency, discount_curve, hazard_curve
)

baseline = pd.Series({
    'Present value': result.present_value,
    'Protection leg': result.protection_leg,
    'Scheduled premium leg': result.scheduled_premium_leg,
    'Accrual on default': result.accrual_on_default,
    'Risky PV01': result.risky_pv01,
    'Model fair spread': result.fair_spread,
    'Contract coupon': contract.coupon,
})
display(baseline.to_frame('Value'))
display(cashflow_detail)

## Part 8 — Parallel and key-rate CS01

CS01 measures the effect of changing market CDS spreads. This notebook bumps the market quotes, **rebootstraps the entire hazard curve**, and fully reprices the trade.

Parallel CS01 is

$$CS01=\frac{V(s+1\text{ bp})-V(s-1\text{ bp})}{2}.$$

It is the signed PV change for a +1 bp parallel spread move. Key-rate CS01 bumps one quoted maturity at a time. Its allocation depends on quote maturities, interpolation, recovery, and bootstrap conventions.


In [ ]:
def price_from_quotes(contract, quotes, discount_curve):
    curve = bootstrap_hazard_curve(quotes, discount_curve)
    return price_cds(contract, discount_curve, curve, quotes.recovery).present_value


def parallel_cs01(contract, quotes, discount_curve, bump_bp=1.0):
    if bump_bp <= 0:
        raise ValueError('bump_bp must be positive.')
    bump = bump_bp / 10_000.0
    up_spreads = tuple(s + bump for s in quotes.par_spreads)
    down_spreads = tuple(s - bump for s in quotes.par_spreads)
    if min(down_spreads) < 0:
        raise ValueError('Down-spread bump produces a negative market quote.')
    pv_up = price_from_quotes(contract, replace(quotes, par_spreads=up_spreads), discount_curve)
    pv_down = price_from_quotes(contract, replace(quotes, par_spreads=down_spreads), discount_curve)
    return (pv_up - pv_down) / (2.0 * bump_bp)


def key_rate_cs01(contract, quotes, discount_curve, bump_bp=1.0):
    if bump_bp <= 0:
        raise ValueError('bump_bp must be positive.')
    bump = bump_bp / 10_000.0
    rows = []
    for index, maturity in enumerate(quotes.maturities):
        up_spreads = list(quotes.par_spreads)
        down_spreads = list(quotes.par_spreads)
        up_spreads[index] += bump
        down_spreads[index] -= bump
        if down_spreads[index] < 0:
            raise ValueError('Down-spread bump produces a negative market quote.')
        pv_up = price_from_quotes(
            contract, replace(quotes, par_spreads=tuple(up_spreads)), discount_curve
        )
        pv_down = price_from_quotes(
            contract, replace(quotes, par_spreads=tuple(down_spreads)), discount_curve
        )
        rows.append({
            'Quote maturity (years)': maturity,
            'Key-rate CS01': (pv_up - pv_down) / (2.0 * bump_bp),
        })
    return pd.DataFrame(rows).set_index('Quote maturity (years)')


parallel_credit_risk = parallel_cs01(contract, market_quotes, discount_curve)
key_rate_credit_risk = key_rate_cs01(contract, market_quotes, discount_curve)
display(pd.Series({'Parallel CS01': parallel_credit_risk}).to_frame('Signed PV change per +1 bp'))
display(key_rate_credit_risk)

## Part 9 — Discount IR01, recovery sensitivity, and jump-to-default

Two conventions are shown for discount and recovery shocks:

- **fixed hazard:** change the selected input while holding calibrated hazard rates unchanged;
- **recalibrated:** change the input and rebootstrap hazards so the original CDS spread quotes are still matched.

The distinction matters because recovery, discounting, and hazard rates jointly determine quoted spreads. Risk reports must state which convention they use.

At a coupon-date valuation with no accrued premium, immediate-default value to a protection buyer is $N(1-R)$. Jump-to-default P&L is immediate-default value minus current model value, with the sign reversed for the protection seller.


In [ ]:
def discount_ir01(contract, quotes, discount_curve, hazard_curve, bump_bp=1.0):
    up_discount = discount_curve.parallel_shift(bump_bp)
    down_discount = discount_curve.parallel_shift(-bump_bp)

    fixed_up = price_cds(contract, up_discount, hazard_curve, quotes.recovery).present_value
    fixed_down = price_cds(contract, down_discount, hazard_curve, quotes.recovery).present_value

    recal_up_curve = bootstrap_hazard_curve(quotes, up_discount)
    recal_down_curve = bootstrap_hazard_curve(quotes, down_discount)
    recal_up = price_cds(contract, up_discount, recal_up_curve, quotes.recovery).present_value
    recal_down = price_cds(contract, down_discount, recal_down_curve, quotes.recovery).present_value

    return (
        (fixed_up - fixed_down) / (2.0 * bump_bp),
        (recal_up - recal_down) / (2.0 * bump_bp),
    )


def recovery_sensitivity(
    contract, quotes, discount_curve, hazard_curve, recovery_point_bump=1.0
):
    bump = recovery_point_bump / 100.0
    recovery_up = quotes.recovery + bump
    recovery_down = quotes.recovery - bump
    if recovery_down < 0 or recovery_up >= 1:
        raise ValueError('Recovery bump leaves the valid interval [0, 1).')

    fixed_up = price_cds(contract, discount_curve, hazard_curve, recovery_up).present_value
    fixed_down = price_cds(contract, discount_curve, hazard_curve, recovery_down).present_value

    up_quotes = replace(quotes, recovery=recovery_up)
    down_quotes = replace(quotes, recovery=recovery_down)
    recal_up = price_from_quotes(contract, up_quotes, discount_curve)
    recal_down = price_from_quotes(contract, down_quotes, discount_curve)

    return (
        (fixed_up - fixed_down) / (2.0 * recovery_point_bump),
        (recal_up - recal_down) / (2.0 * recovery_point_bump),
    )


def jump_to_default(contract, current_pv, recovery):
    side_sign = 1.0 if contract.side.lower() == 'protection_buyer' else -1.0
    immediate_default_value = side_sign * contract.notional * (1.0 - recovery)
    return immediate_default_value - current_pv


fixed_ir01, recalibrated_ir01 = discount_ir01(
    contract, market_quotes, discount_curve, hazard_curve
)
fixed_recovery, recalibrated_recovery = recovery_sensitivity(
    contract, market_quotes, discount_curve, hazard_curve
)
jtd = jump_to_default(contract, result.present_value, market_quotes.recovery)

other_risk = pd.DataFrame([
    {'Risk measure': 'Discount IR01 — fixed hazard', 'Value': fixed_ir01, 'Unit': 'PV per +1 bp discount shift'},
    {'Risk measure': 'Discount IR01 — recalibrated', 'Value': recalibrated_ir01, 'Unit': 'PV per +1 bp discount shift'},
    {'Risk measure': 'Recovery sensitivity — fixed hazard', 'Value': fixed_recovery, 'Unit': 'PV per +1 recovery percentage point'},
    {'Risk measure': 'Recovery sensitivity — recalibrated', 'Value': recalibrated_recovery, 'Unit': 'PV per +1 recovery percentage point'},
    {'Risk measure': 'Jump-to-default', 'Value': jtd, 'Unit': 'Immediate-default P&L'},
]).set_index('Risk measure')
display(other_risk)

## Part 10 — Full-revaluation scenarios

Scenario analysis shocks quoted spreads, discount rates, and recovery together. The hazard curve is then rebootstrapped from the shocked market state before the trade is repriced.

For a protection buyer:

- wider spreads normally increase value;
- lower recovery normally increases the default payment;
- combined shocks need full revaluation because the calibrated hazard curve changes as well.


In [ ]:
@dataclass(frozen=True)
class CDSScenario:
    name: str
    spread_shift_bp: float = 0.0
    discount_shift_bp: float = 0.0
    recovery_shift: float = 0.0


def cds_scenario_report(contract, quotes, discount_curve, scenarios):
    base_pv = price_from_quotes(contract, quotes, discount_curve)
    rows = []
    for scenario in scenarios:
        shocked_spreads = tuple(
            spread + scenario.spread_shift_bp / 10_000.0
            for spread in quotes.par_spreads
        )
        shocked_recovery = quotes.recovery + scenario.recovery_shift
        if min(shocked_spreads) < 0:
            raise ValueError(f'Scenario {scenario.name!r} produces a negative CDS spread.')
        if not 0.0 <= shocked_recovery < 1.0:
            raise ValueError(f'Scenario {scenario.name!r} produces invalid recovery.')
        shocked_quotes = replace(
            quotes, par_spreads=shocked_spreads, recovery=shocked_recovery
        )
        shocked_discount = discount_curve.parallel_shift(scenario.discount_shift_bp)
        shocked_curve = bootstrap_hazard_curve(shocked_quotes, shocked_discount)
        shocked_result = price_cds(
            contract, shocked_discount, shocked_curve, shocked_recovery
        )
        rows.append({
            'Scenario': scenario.name,
            'Spread shift (bp)': scenario.spread_shift_bp,
            'Discount shift (bp)': scenario.discount_shift_bp,
            'Recovery': shocked_recovery,
            'Fair spread': shocked_result.fair_spread,
            'Present value': shocked_result.present_value,
            'Full-revaluation P&L': shocked_result.present_value - base_pv,
        })
    return pd.DataFrame(rows).set_index('Scenario')


scenarios = [
    CDSScenario('Base'),
    CDSScenario('Spreads +100 bp', spread_shift_bp=100.0),
    CDSScenario('Spreads -50 bp', spread_shift_bp=-50.0),
    CDSScenario('Recovery -10 points', recovery_shift=-0.10),
    CDSScenario('Discount curve +100 bp', discount_shift_bp=100.0),
    CDSScenario('Credit deterioration', spread_shift_bp=250.0, discount_shift_bp=-50.0, recovery_shift=-0.20),
    CDSScenario('Credit improvement', spread_shift_bp=-50.0, discount_shift_bp=50.0, recovery_shift=0.10),
]

scenario_results = cds_scenario_report(
    contract, market_quotes, discount_curve, scenarios
)
display(scenario_results)

## Part 11 — Numerical and theoretical validation

The validation block checks:

1. every bootstrapped maturity reprices to its market par spread;
2. a CDS struck at its calibrated par spread has approximately zero value;
3. protection-buyer and protection-seller values are exact opposites;
4. survival probabilities remain inside $[0,1]$ and decrease through time;
5. widening all market spreads increases the value of fixed-coupon protection;
6. protection and premium-leg values remain non-negative.

Passing these checks is necessary but not sufficient for production. Independent library benchmarking and market-outcome monitoring are still required.


In [ ]:
calibration_rows = []
for maturity, market_spread in zip(market_quotes.maturities, market_quotes.par_spreads):
    model_spread = cds_par_spread(
        maturity, market_quotes.payment_frequency, discount_curve, hazard_curve, market_quotes.recovery
    )
    calibration_rows.append({
        'Maturity': maturity,
        'Market spread': market_spread,
        'Model spread': model_spread,
        'Error (bp)': 10_000.0 * (model_spread - market_spread),
    })
calibration_check = pd.DataFrame(calibration_rows)
assert calibration_check['Error (bp)'].abs().max() < 1e-8

# A five-year par CDS should have approximately zero PV.
par_contract = replace(contract, coupon=market_quotes.par_spreads[2])
par_pv = price_cds(par_contract, discount_curve, hazard_curve, market_quotes.recovery).present_value
assert abs(par_pv) < 1e-5

# Buyer/seller antisymmetry.
seller_contract = replace(contract, side='protection_seller')
seller_pv = price_cds(seller_contract, discount_curve, hazard_curve, market_quotes.recovery).present_value
assert abs(result.present_value + seller_pv) < 1e-10

# Survival probabilities must be valid and non-increasing.
validation_times = np.linspace(0.0, market_quotes.maturities[-1], 401)
survival = hazard_curve.survival(validation_times)
assert np.all((survival >= 0.0) & (survival <= 1.0))
assert np.all(np.diff(survival) <= 1e-14)

# A parallel spread widening must increase this protection buyer's value.
wide_quotes = replace(
    market_quotes,
    par_spreads=tuple(s + 0.0010 for s in market_quotes.par_spreads),
)
wide_pv = price_from_quotes(contract, wide_quotes, discount_curve)
assert wide_pv > result.present_value
assert result.protection_leg >= 0.0
assert result.scheduled_premium_leg >= 0.0
assert result.accrual_on_default >= 0.0

display(calibration_check)
validation_summary = pd.Series({
    'Five-year par CDS PV': par_pv,
    'Buyer plus seller PV': result.present_value + seller_pv,
    'Minimum survival probability': float(np.min(survival)),
    'Base protection-buyer PV': result.present_value,
    'PV after parallel +10 bp spread move': wide_pv,
})
display(validation_summary.to_frame('Value'))
print('All core CDS validation checks passed.')

## Part 12 — Market data needed for production use

The illustrative inputs are sufficient for learning and testing. A market-calibrated implementation requires:

1. valuation date, trade date, effective date, maturity date, and reporting currency;
2. reference entity and seniority;
3. protection buyer/seller direction, notional, contractual coupon, and any upfront amount;
4. CDS documentation and restructuring convention;
5. coupon dates, IMM rules, accrual day count, calendars, business-day adjustments, and settlement lag;
6. collateral-consistent discount curve and its bootstrapping instruments;
7. CDS market quotes by maturity, including whether each quote is par spread, upfront, or price;
8. standard coupon convention and accrued premium treatment;
9. approved recovery assumption and recovery-risk methodology;
10. bid/mid/ask data, stale-price controls, and source timestamps;
11. counterparty and wrong-way-risk inputs if valuation adjustments are in scope.


## Part 13 — Controlled next extensions

After the simplified model is understood and independently validated, the recommended development order is:

1. date-based CDS schedules, IMM dates, day counts, calendars, and settlement delays;
2. exact integration of default payments and accrual on default within each interval;
3. standard-coupon CDS pricing with upfront amounts and clean/dirty price conventions;
4. discount-curve bootstrapping and credit-quote interpolation controls;
5. carry, roll-down, realized P&L explain, and aging of the credit curve;
6. bucketed IR01, recovery scenarios, and liquidity/bid-ask reserves;
7. index CDS, index factors, and constituent-default treatment;
8. CDS options using spread or hazard-rate volatility models;
9. counterparty credit valuation adjustment and wrong-way risk;
10. independent benchmarking, limits, backtesting, and model governance.
